# 09 · İçerik Çıkarımının Elle Değerlendirilmesi

Sistemin test setinden 20 belgeden (her türden 5) çıkardığı **110 alan** tek tek gösterilir. Solda belgenin resmi, sağda sistemin çıkardığı alanlar var.
Hücredeki **CEVAP** kutusuna belge numarası ve her alan için bir harf (d/y/k/b) yazıp Enter'a basarsın.

| Seçenek | Ne zaman |
|---|---|
| **Doğru** | Çıkarılan değer belgede yazanla aynı (küçük OCR harf hatası olsa da doğru say) |
| **Yanlış** | Değer hatalı ya da başka bir alanın bilgisi (ör. ad yerine "BIOGRAPHICAL SKETCH") |
| **Kaçırıldı** | Sistem "bulunamadı" dedi ama bilgi belgede var |
| **Belgede yok** | Sistem "bulunamadı" dedi ve bilgi belgede gerçekten yok |

- Seçimler her belgede Drive'a kaydedilir (`bitirme_ocr/cikarim_degerlendirme_v2.csv`). Yarıda bırakırsan tekrar çalıştırınca kaldığın belgeden devam eder.
- En sondaki hücre doğruluk oranlarını hesaplar.
- OCR ya da model gerekmez; sadece Drive'daki resimler kullanılır. Açılış ~30 sn.

Bu 110 alan, raporun **Elle değerlendirme** sekmesindekilerle aynıdır; ikisinden birini doldurman yeterli.

## 1. Drive ve veriler

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, io
import pandas as pd
from PIL import Image

ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA = ROOT / "data"
OUT = ROOT / "cikarim_degerlendirme_v2.csv"
TR = {"email": "E-posta", "invoice": "Fatura", "resume": "CV", "scientific_publication": "Bilimsel makale"}

In [ ]:
ROWS = [
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "gonderen",
"deger": "Ostern, Eric on Thu, Apr 4, 1996 10:45 AM"
},
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "alici",
"deger": "Portnoy, Sharon"
},
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "konu",
"deger": "FW: NYC Rest. Contacts"
},
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "tarih",
"deger": "Apr 4, 1996"
},
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "e_postalar",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "govde_ozeti",
"deger": "Ce: Bayliss, Elissa; Culley, Liz; Young, Tracy Hi Sharon: Thad Elissa follow-up on obtaining info from the restaurants b"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "gonderen",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "alici",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "konu",
"deger": "Washington Post Story"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "tarih",
"deger": "April 29, 1998"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "e_postalar",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "govde_ozeti",
"deger": "Carothers, Paul Wednesday, April 29, 1998 6:07 PM Knuth, Kathy S. (MS Mail); Dennis, Darienne; Ostronic, John;"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "gonderen",
"deger": "ebh@caunxi.ctd.crnl.gov (B4 Harris)"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "alici",
"deger": "cvrGdsunix?.derd.crnl.gov"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "konu",
"deger": "Tobacco Funding Question"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "tarih",
"deger": "Tue, 26 Mar 1996 17:40:32 -0500"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "e_postalar",
"deger": "ebh@caunxi.ctd.crnl.gov, sikkavk@dsunix2.dsrd.ornl.cov, _vks@ornl.gov"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "govde_ozeti",
"deger": "Ed_Harris,3/26/96 4:40 PM,Tobacco Funding Question ce: sikkavk@dsunix2.dsrd.ornl.cov vee,"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "gonderen",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "alici",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "konu",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "tarih",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "e_postalar",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "govde_ozeti",
"deger": "52021 9188 yO 8 Buunjoeynuey spuesa \"YO \"BuURDeINUEN SUSHPOUIERD MUAANG"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "gonderen",
"deger": "Date:"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "alici",
"deger": "co:"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "konu",
"deger": "To:"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "tarih",
"deger": "Subject:"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "e_postalar",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "govde_ozeti",
"deger": "co: 2071557599 Note for Knuff, Ketrina"
},
{
"belge": "images/test/invoice/test_invoice_00263.png",
"tur": "invoice",
"alan": "fatura_no",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00263.png",
"tur": "invoice",
"alan": "tarih",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00263.png",
"tur": "invoice",
"alan": "firma",
"deger": "PHILIP MORRIS MANAGEMENT CORPORATION"
},
{
"belge": "images/test/invoice/test_invoice_00263.png",
"tur": "invoice",
"alan": "toplam_tutar",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00263.png",
"tur": "invoice",
"alan": "tutarlar",
"deger": "1723.58"
},
{
"belge": "images/test/invoice/test_invoice_00241.png",
"tur": "invoice",
"alan": "fatura_no",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00241.png",
"tur": "invoice",
"alan": "tarih",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00241.png",
"tur": "invoice",
"alan": "firma",
"deger": "WoRsN"
},
{
"belge": "images/test/invoice/test_invoice_00241.png",
"tur": "invoice",
"alan": "toplam_tutar",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00241.png",
"tur": "invoice",
"alan": "tutarlar",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00296.png",
"tur": "invoice",
"alan": "fatura_no",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00296.png",
"tur": "invoice",
"alan": "tarih",
"deger": "1/24/89"
},
{
"belge": "images/test/invoice/test_invoice_00296.png",
"tur": "invoice",
"alan": "firma",
"deger": "PROMOTIONAL MARKETING, INC"
},
{
"belge": "images/test/invoice/test_invoice_00296.png",
"tur": "invoice",
"alan": "toplam_tutar",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00296.png",
"tur": "invoice",
"alan": "tutarlar",
"deger": "$25,500, $17,400,, $37,950"
},
{
"belge": "images/test/invoice/test_invoice_00218.png",
"tur": "invoice",
"alan": "fatura_no",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00218.png",
"tur": "invoice",
"alan": "tarih",
"deger": "02/18/94"
},
{
"belge": "images/test/invoice/test_invoice_00218.png",
"tur": "invoice",
"alan": "firma",
"deger": "Brand Precielon Services, Inc. 1so0a761 it"
},
{
"belge": "images/test/invoice/test_invoice_00218.png",
"tur": "invoice",
"alan": "toplam_tutar",
"deger": "1,130.00"
},
{
"belge": "images/test/invoice/test_invoice_00218.png",
"tur": "invoice",
"alan": "tutarlar",
"deger": "10.00 , 50.00 , 500.00 , 10.00 , 22.00"
},
{
"belge": "images/test/invoice/test_invoice_00298.png",
"tur": "invoice",
"alan": "fatura_no",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00298.png",
"tur": "invoice",
"alan": "tarih",
"deger": "November 8 00"
},
{
"belge": "images/test/invoice/test_invoice_00298.png",
"tur": "invoice",
"alan": "firma",
"deger": "ANYNEX Company"
},
{
"belge": "images/test/invoice/test_invoice_00298.png",
"tur": "invoice",
"alan": "toplam_tutar",
"deger": "$956.83"
},
{
"belge": "images/test/invoice/test_invoice_00298.png",
"tur": "invoice",
"alan": "tutarlar",
"deger": "$956.83, 9638.67 , 5638.67 , $5.10, 313.06"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "ad_soyad",
"deger": "BIOGRAPHICAL SKETCH"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "universite",
"deger": "Tel-Aviv University, Tel-Aviv, Israel B.Sc. [1966-1969 Biology"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "bolum_derece",
"deger": "Tel-Aviv University, Tel-Aviv, Israel Ph.D. [1973-1977 Endocrinology of Reproductic:"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "e_posta",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "telefon",
"deger": "30502-30509"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "beceriler",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "ad_soyad",
"deger": "EDWARD DIALA"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "universite",
"deger": "Harvard University Special Student in Biology 1972-4"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "bolum_derece",
"deger": "Institution Subject & Degree Year"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "e_posta",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "telefon",
"deger": "50145702"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "beceriler",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "ad_soyad",
"deger": "ANN KATHERINE DALY"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "universite",
"deger": "BA in Biochemistry (1978, University of Dublin); PhD in Biochemistry (1982,"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "bolum_derece",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "e_posta",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "telefon",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "beceriler",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "ad_soyad",
"deger": "Cecilia M. Giachelli Research"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "universite",
"deger": "University of Washington, Seattle, WA Phb, 1987 Pharmacolo"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "bolum_derece",
"deger": "American Association for the Advancement of Science"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "e_posta",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "telefon",
"deger": "50304562"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "beceriler",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "ad_soyad",
"deger": "Pose Bs"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "universite",
"deger": "Metabolism, Baylor College of Medicine"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "bolum_derece",
"deger": "Key personnel typical wil include all Indviduls wih doctoral or other professional degrees, btn some projects wil includ"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "e_posta",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "telefon",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "beceriler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00046.png",
"tur": "scientific_publication",
"alan": "baslik",
"deger": "Printed i Great Brin"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00046.png",
"tur": "scientific_publication",
"alan": "yazarlar",
"deger": "DAVID 3"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00046.png",
"tur": "scientific_publication",
"alan": "ozet",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00046.png",
"tur": "scientific_publication",
"alan": "anahtar_kelimeler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00046.png",
"tur": "scientific_publication",
"alan": "kaynakca_var",
"deger": "False"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00057.png",
"tur": "scientific_publication",
"alan": "baslik",
"deger": "What the Physician Should Know About Tobacco"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00057.png",
"tur": "scientific_publication",
"alan": "yazarlar",
"deger": "WALTER A. BASTEDO, tobaccos"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00057.png",
"tur": "scientific_publication",
"alan": "ozet",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00057.png",
"tur": "scientific_publication",
"alan": "anahtar_kelimeler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00057.png",
"tur": "scientific_publication",
"alan": "kaynakca_var",
"deger": "False"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00030.png",
"tur": "scientific_publication",
"alan": "baslik",
"deger": "Sondereudh us: eitge to Tabklordhng\" Band 8- Heb 7 Ohtake 976"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00030.png",
"tur": "scientific_publication",
"alan": "yazarlar",
"deger": "Cytrel, Dav Git Vidroy"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00030.png",
"tur": "scientific_publication",
"alan": "ozet",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00030.png",
"tur": "scientific_publication",
"alan": "anahtar_kelimeler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00030.png",
"tur": "scientific_publication",
"alan": "kaynakca_var",
"deger": "False"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00012.png",
"tur": "scientific_publication",
"alan": "baslik",
"deger": "PN LEE, w+ FIR RED Ntel cit Gees"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00012.png",
"tur": "scientific_publication",
"alan": "yazarlar",
"deger": "LEE, GP Se, Sretrcewn, Wie Lis"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00012.png",
"tur": "scientific_publication",
"alan": "ozet",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00012.png",
"tur": "scientific_publication",
"alan": "anahtar_kelimeler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00012.png",
"tur": "scientific_publication",
"alan": "kaynakca_var",
"deger": "False"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00051.png",
"tur": "scientific_publication",
"alan": "baslik",
"deger": "TOBACCO SHOKE AND THE PULMONARY ALVEOLAR MACROPHAGE"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00051.png",
"tur": "scientific_publication",
"alan": "yazarlar",
"deger": "DRATH, P. DAVIES, M. L. KARNOVSKY, Beth Israel Hospital Boston, Teceresrtct"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00051.png",
"tur": "scientific_publication",
"alan": "ozet",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00051.png",
"tur": "scientific_publication",
"alan": "anahtar_kelimeler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00051.png",
"tur": "scientific_publication",
"alan": "kaynakca_var",
"deger": "False"
}
]

ev = pd.DataFrame(ROWS)
ev["deger"] = ev["deger"].replace({"None": "— (bulunamadı)"}).str.replace(r"\s+", " ", regex=True)
docs = list(dict.fromkeys(ev["belge"]))          # sıra korunur
done = pd.read_csv(OUT) if OUT.exists() else pd.DataFrame(columns=["belge", "tur", "alan", "deger", "karar"])
print(f"{len(docs)} belge, {len(ev)} alan · daha önce değerlendirilen: {done['belge'].nunique()} belge")

## 2. Değerlendirme

Aşağıdaki hücrede **CEVAP** kutusu var. Belgenin altında yazan **belge numarasını ve her alan için bir harfi** yaz, Enter'a bas. Hücre kendiliğinden çalışır, cevabı kaydeder ve sonraki belgeyi getirir.

| Harf | Anlamı |
|---|---|
| **d** | Doğru |
| **y** | Yanlış |
| **k** | Kaçırıldı (sistem bulamadı ama belgede var) |
| **b** | Belgede yok (sistem bulamadı, belgede de yok) |

Örnek: Belge 1'de 6 alan varsa kutuya `1 ddydkb` yaz ve Enter.
- Önceki bir belgeyi görmek/düzeltmek için sadece numarasını yaz (ör. `3`), sonra `3 dddyk` gibi yeniden cevapla.
- Her cevap Drive'a hemen kaydedilir; kapatıp açınca kaldığın yerden devam eder.

In [ ]:
#@title Belge değerlendirme · CEVAP kutusuna "belge_no harfler" yaz, Enter { run: "auto", display-mode: "form" }
CEVAP = "" #@param {type:"string"}

from IPython.display import display, Image as IImage

HARF = {"d": "Doğru", "y": "Yanlış", "k": "Kaçırıldı", "b": "Belgede yok"}

def kaydet(belge, kararlar):
    global done
    if "belge" not in done.columns:
        done = pd.DataFrame(columns=["belge", "tur", "alan", "deger", "karar"])
    sub = ev[ev.belge == belge].copy()
    sub["karar"] = kararlar
    done = pd.concat([done[done.belge != belge], sub], ignore_index=True)
    done.to_csv(OUT, index=False)

def goster(i, not_=""):
    belge = docs[i]
    sub = ev[ev.belge == belge].reset_index(drop=True)
    if not_: print(not_ + "\n")
    print(f"Belge {i + 1} / {len(docs)} · {TR.get(sub.tur[0], sub.tur[0])} · {Path(belge).name}"
          f" · değerlendirilen: {done['belge'].nunique()}/{len(docs)}")
    im = Image.open(DATA / belge).convert("L"); im.thumbnail((700, 900))
    buf = io.BytesIO(); im.save(buf, format="PNG")
    display(IImage(data=buf.getvalue(), width=480))
    print("Sistemin çıkardığı alanlar:")
    for n, r in sub.iterrows():
        deger = r.deger if len(r.deger) <= 55 else r.deger[:52] + "..."
        print(f"  {n + 1}. {r.alan}: {deger}")
    print(f"\n>>> CEVAP kutusuna yaz:  {i + 1} " + "?" * len(sub) + f"   ({len(sub)} harf: d / y / k / b)")

def siradaki():
    yapilan = set(done["belge"]) if "belge" in done.columns else set()
    return next((k for k, d in enumerate(docs) if d not in yapilan), None)

parca = CEVAP.strip().lower().split()
not_ = ""
if parca and parca[0].isdigit() and 1 <= int(parca[0]) <= len(docs):
    i = int(parca[0]) - 1
    n_alan = len(ev[ev.belge == docs[i]])
    harfler = "".join(parca[1:])
    if not harfler:
        goster(i, f"Belge {i + 1} gösteriliyor (düzeltmek için: {i + 1} ve harfler).")
        i = None
    elif len(harfler) == n_alan and set(harfler) <= set(HARF):
        kaydet(docs[i], [HARF[c] for c in harfler])
        not_ = f"✓ Belge {i + 1} kaydedildi ({harfler})."
        i = siradaki()
    else:
        goster(i, f"!! Belge {i + 1} için {n_alan} harf gerekiyor, sadece d/y/k/b (yazdığın: '{harfler}'). Tekrar yaz.")
        i = None
elif parca:
    not_ = f"!! Anlaşılmadı: '{CEVAP}'. Örnek: 1 ddydkb"
    i = siradaki()
else:
    i = siradaki()

if i is not None:
    goster(i, not_)
elif not parca or (parca and parca[0].isdigit() and len(parca) > 1 and siradaki() is None):
    print(not_ + f"\nBitti! {done['belge'].nunique()} belge değerlendirildi. Şimdi aşağıdaki 3. bölümün hücresini çalıştır.")

## 3. Sonuçlar

Değerlendirmeyi bitirince (ya da istediğin zaman) bu hücreyi çalıştır.

In [ ]:
res = pd.read_csv(OUT) if OUT.exists() else pd.DataFrame()   # 'done' ekranın tablosu; burada ayrı ad
if res.empty:
    print("Henüz değerlendirme yok.")
else:
    def score(g):
        c = g["karar"].value_counts()
        d, y, k, b = (int(c.get(x, 0)) for x in ["Doğru", "Yanlış", "Kaçırıldı", "Belgede yok"])
        return pd.Series({"alan": len(g), "doğru": d, "yanlış": y, "kaçırıldı": k, "belgede yok": b,
                          "precision": round(d / (d + y), 3) if d + y else None,   # çıkardığının ne kadarı doğru
                          "recall": round(d / (d + k), 3) if d + k else None,      # belgede olanın ne kadarını buldu
                          "doğruluk": round((d + b) / len(g), 3)})                 # tüm kararlar içinde doğru davranma
    by_type = res.groupby("tur").apply(score)
    by_type.loc["TOPLAM"] = score(res)
    by_field = res.groupby(["tur", "alan"]).apply(score)
    print(f"Değerlendirilen: {res['belge'].nunique()} belge, {len(res)} alan\n")
    print("Türe göre:"); display(by_type)
    print("Alana göre:"); display(by_field)
    by_field.to_csv(ROOT / "cikarim_sonuclari_v2.csv")
    by_type.to_csv(ROOT / "cikarim_sonuclari_tur_v2.csv")
    print("Kaydedildi:", ROOT / "cikarim_sonuclari_v2.csv")

| Ölçüt | Anlamı |
|---|---|
| **Precision** | Sistemin bir şey çıkardığı alanların ne kadarı doğru |
| **Recall** | Belgede gerçekten olan bilgilerin ne kadarını sistem bulabildi |
| **Doğruluk** | "Belgede yok" doğru tespitleri de sayılınca, alanların ne kadarında sistem doğru davrandı |

## 4. İnceleme PDF'i (etkileşimsiz)

Kutu/düğme ile uğraşmak istemezsen bu hücreyi çalıştır. 20 belgenin her biri için bir sayfa hazırlar: **belgenin resmi**, sistemin **çıkardığı alanlar** ve **OCR'ın okuduğu metin**. PDF Drive'a kaydedilir: `bitirme_ocr/cikarim_inceleme_20belge.pdf`. Kararlarını raporun **Elle değerlendirme** sekmesindeki açılır listelere girebilirsin (aynı 110 alan, aynı sıra).

In [ ]:
import textwrap
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

ocr = pd.concat([pd.read_csv(DATA / f) for f in ["ocr_dataset.csv", "ocr_dataset_aday.csv"] if (DATA / f).exists()])
ocr["ad"] = ocr["path"].astype(str).map(lambda x: Path(x).name)
OCR_TEXT = dict(zip(ocr["ad"], ocr["text"].fillna("")))
PDF = ROOT / "cikarim_inceleme_20belge.pdf"

def sar(t, n):
    return "\n".join(textwrap.fill(x, n) if x.strip() else "" for x in str(t).splitlines())

with PdfPages(PDF) as pdf:
    f = plt.figure(figsize=(8.27, 11.69)); f.text(0.08, 0.92, "İçerik çıkarımı · 20 test belgesi (110 alan)", fontsize=16, weight="bold")
    f.text(0.08, 0.86, sar("Her sayfada: solda belgenin resmi, sağda sistemin çıkardığı alanlar, altta OCR'ın okuduğu metnin başı. "
                           "Her alan için karar ver ve raporun 'Elle değerlendirme' sekmesine gir.", 90), fontsize=10, va="top")
    f.text(0.08, 0.76, "Doğru       : çıkarılan değer belgede yazanla aynı (küçük harf hatası olsa da)\n"
                       "Yanlış      : değer hatalı ya da başka bir alanın bilgisi\n"
                       "Kaçırıldı   : sistem 'bulunamadı' dedi ama bilgi belgede var\n"
                       "Belgede yok : sistem 'bulunamadı' dedi ve bilgi belgede gerçekten yok", fontsize=10, family="monospace", va="top")
    pdf.savefig(f); plt.close(f)
    for i, belge in enumerate(docs):
        sub = ev[ev.belge == belge].reset_index(drop=True)
        f = plt.figure(figsize=(8.27, 11.69))
        f.text(0.04, 0.975, f"Belge {i + 1} / {len(docs)} · {TR.get(sub.tur[0], sub.tur[0])} · {Path(belge).name}", fontsize=12, weight="bold", va="top")
        ax = f.add_axes([0.03, 0.30, 0.50, 0.65]); ax.imshow(Image.open(DATA / belge).convert("L"), cmap="gray", vmin=0, vmax=255); ax.axis("off")
        lines = ["SİSTEMİN ÇIKARDIĞI ALANLAR", ""]
        for n, r in sub.iterrows():
            lines += [f"{n + 1}. {r.alan}", textwrap.indent(sar(r.deger, 34), "    "), "    karar: ______________", ""]
        f.text(0.56, 0.94, "\n".join(lines), fontsize=8, family="monospace", va="top")
        metin = OCR_TEXT.get(Path(belge).name, "(OCR metni bulunamadı)")
        metin = "\n".join(sar(metin, 120).splitlines()[:28])
        f.text(0.04, 0.28, "OCR'IN OKUDUĞU METİN (ilk satırlar)", fontsize=9, weight="bold", va="top")
        f.text(0.04, 0.265, metin, fontsize=6.5, family="monospace", va="top")
        pdf.savefig(f); plt.close(f)
print("Kaydedildi:", PDF, f"({len(docs) + 1} sayfa)")